# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset describing extracellular vesicle mass spectrometry analysis using the `mlcroissant` library.

### Dataset Source

The dataset is provided as a Croissant schema:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

We load the dataset metadata and examine its main details.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata with mlcroissant
dataset = mlc.Dataset(url)

print(f"Dataset loaded: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview

Explore the available record sets, their `@id`, and fields. We enumerate all available record sets in this Croissant package, then for each record set, their field (column) `@id` and names.

In [ ]:
# List available record sets, fields, and columns by their @id
print("Available record sets:")
for record_set in dataset.metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id} | Name: {getattr(record_set, 'name', '')}")
    if hasattr(record_set, 'fields'):
        for field in record_set.fields:
            col_name = getattr(field, 'name', '')
            print(f"    - Field @id: {field.id} | Name: {col_name} | Schema DataType: {getattr(field, 'data_type', '')}")

## 3. Data Extraction

We load the main protein abundance table for analysis.

**Note:** We use the `@id` of the record set. Often the main table is the first record set listed.  
Assign the record set `@id` you want to process below.

We build a dictionary of DataFrames, keyed by record set `@id`.  
Fields are accessible as DataFrame columns. All identifiers are referenced by their Croissant schema `@id`.

In [ ]:
# Get the list of record set @ids
record_sets = [r.id for r in dataset.metadata.record_sets]

dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set: {record_set_id} - Shape: {df.shape}")

# Pick the main protein table (assume first)
main_recordset_id = record_sets[0]
print(f"\nColumns in {main_recordset_id}:")
print(dataframes[main_recordset_id].columns.tolist())

# Show first few rows
dataframes[main_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)

This section demonstrates basic filtering, normalization, and grouping operations.

- We select a numeric field (by `@id`) from the record set, e.g., peptide count, abundance, coverage_percent, or molecular_weight.
- Filter for records where the field exceeds a threshold.
- Normalize the field (z-score normalization).
- Optionally, group by a categorical field (e.g., protein description or evidence class).

⚠️ *Update `numeric_field_id` and `group_field_id` appropriately, referencing the correct `@id` as listed above!*

In [ ]:
# Example: Suppose the main table uses '@id: ProteinTable', numeric field '@id: PeptideCount', group field '@id: Description' - replace with real values as printed above

record_set_id = main_recordset_id  # e.g., '@id' of main protein table

# Pick numeric field: search above for field with data_type 'Float' or 'Integer', get @id
# For illustration, use 'Peptide count' if it exists, otherwise pick another numeric.

candidate_numeric_fields = [f.id for f in dataset.metadata.record_sets[0].fields if getattr(f, 'data_type', None) in ['Float', 'Integer', 'Number']]
print("Candidate numeric field ids:", candidate_numeric_fields)
numeric_field_id = candidate_numeric_fields[0]

threshold = 10
# Only keep rows where the numeric field is not null and > threshold
df = dataframes[record_set_id]
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
print(f"\nWith normalized {numeric_field_id} (z-score):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping field, pick a textual/categorical field
candidate_group_fields = [f.id for f in dataset.metadata.record_sets[0].fields if getattr(f, 'data_type', None) in ['Text', 'String']]
group_field_id = candidate_group_fields[0] if candidate_group_fields else None

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"\nGrouped data by {group_field_id}, mean of {numeric_field_id}:")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the filtered numeric field and its normalized distribution.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 5))
sns.histplot(filtered_df[numeric_field_id].astype(float), bins=30, kde=True, color='steelblue')
plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(12, 5))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=30, kde=True, color='orange')
plt.title(f"Normalized {numeric_field_id} (z-score)")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.ylabel("Count")
plt.show()

if group_field_id and group_field_id in filtered_df.columns:
    # Show mean per group for top groups
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10, 6))
    sns.barplot(y=grouped_df.index, x=grouped_df.values, orient='h', palette='Blues_d')
    plt.title(f"Top 10 {group_field_id} by mean {numeric_field_id}")
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we have:
- Loaded and explored the FAIR² mass spectrometry dataset using the `mlcroissant` library
- Inspected available record sets and fields, referencing all by their `@id`
- Carried out basic EDA including filtering, normalization, and grouping
- Visualized numeric field distributions

**For more information, review the dataset's full Croissant schema and the [mlcroissant documentation](https://mlcommonspublicstorage.blob.core.windows.net/public/language/croissant/1.0/docs/index.html).**